In [ ]:
# ==========================================
# OPTIONAL: Install required libraries
# ==========================================
# !pip install -q -r requirements.txt

In [ ]:
 !pip install -q catboost cleanlab optuna

In [ ]:
#Default Imports
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import randint, loguniform, uniform
import gc
from sklearn.base import clone
import re
from collections import Counter
import json
import joblib
import os
import time
import sys

# mport order is required for MICE
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

#variance checks
from sklearn.feature_selection import VarianceThreshold

#unsupervised patterns extractor
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

#visualization
import seaborn as sns
import matplotlib.pyplot as plt

#pipeline preprocessing
from imblearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE

#model performance
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict,train_test_split
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix

#modelling
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

#cleaning noisy labels
from cleanlab.filter import find_label_issues

#optuna tuning
import optuna

import warnings
warnings.filterwarnings('ignore')



In [ ]:
start = time.time()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#paths
exp_code = str(round(time.time() * 1000))
print(exp_code)

main_path = '/content/drive/MyDrive/FHI_Prediction/'
data_path = '/content/drive/MyDrive/FHI_Prediction/Data/'
models_path = main_path + 'FHI_Models/'
submission_path = models_path + f'{exp_code}/'
raw_models_path = submission_path + 'raw_models/'
clean_models_path = submission_path + 'cleaned_models/'

#create the directories
os.makedirs( models_path, exist_ok=True)
os.makedirs( submission_path, exist_ok=True)
os.makedirs( raw_models_path, exist_ok=True)
os.makedirs( clean_models_path, exist_ok=True)

In [ ]:
#import custom pipelines
sys.path.append(main_path)
from Custom_Transformers.custom_engineering import FHIBaseSignals,FHIAdvancedSignals,DataCleaner,FeatureNameSanitizer,custom_smote_ratios,DropColumns
from Custom_Transformers.custom_patterns import UnsupervisedPatternExtractor
from Custom_Transformers.custom_preprocessing import FHIDataCleaner

In [ ]:
import random
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# 1.LOAD DATA

In [ ]:
train_data = pd.read_csv(data_path + "Train.csv")
test_data = pd.read_csv(data_path + "Test.csv")

In [ ]:
def count_plot(dataset,ax):
    total = len(dataset)
    for p in ax.patches:
        percentage = f'{100 * p.get_height() / total:.1f}%\n'
        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        ax.annotate(percentage, (x, y), ha='center', va='center')
    plt.tight_layout()
    plt.show()

In [ ]:
ax = sns.countplot(data = train_data,x='Target')
count_plot(train_data,ax)

In [ ]:
train_data.columns

# 2. MINIMAL PREPROCESSING

 Majority of the pre-processing and feature engineering is handled by the custom transformers

In [ ]:
#calculate a common age column
train_data['business_age'] = (train_data['business_age_years'].fillna(0))*12 + (train_data['business_age_months'].fillna(0))
test_data['business_age'] = (test_data['business_age_years'].fillna(0))*12 + (test_data['business_age_months'].fillna(0))

###
train_data['business_age'] = train_data['business_age']/12
test_data['business_age'] = test_data['business_age']/12

#drop the years and months colum
train_data = train_data.drop(columns=['business_age_years', 'business_age_months'])
test_data = test_data.drop(columns=['business_age_years', 'business_age_months'])


#3.  MODELLING

## 3.1 ERROR ANALYSIS

In [ ]:
def perform_error_analysis(X_val_df, y_val_true, val_preds, val_probs):
    """
    Conducts a deep dive into the prediction errors
    Expects X_val_df to be a Pandas DataFrame so we can read the column names.
    """
    print("\n" + "="*50)
    print("INITIATING  ERROR ANALYSIS")
    print("="*50)

    # 1. Build the Error DataFrame
    error_df = X_val_df.copy()
    error_df['True_Target'] = y_val_true
    error_df['Pred_Target'] = val_preds
    error_df['Pred_Prob_Low'] = val_probs[:, 1]
    error_df['Pred_Prob_Medium'] = val_probs[:, 2]

    # Extract probabilities (Class 0 is High, Class 1 is Low, Class 2 is Medium)
    error_df['Pred_Prob_Low'] = val_probs[:, 1]
    error_df['Pred_Prob_Medium'] = val_probs[:, 2]
    error_df['Pred_Prob_High'] = val_probs[:, 0]

    #saving all predictions
    error_df.to_csv(submission_path + f"error_analysis_ALL_data.csv", index=False)

    # 2. Isolate the exact errors (True Medium = 2, Predicted Low = 1)
    medium_as_low = error_df[(error_df['True_Target'] == 2) & (error_df['Pred_Target'] == 1)].copy()
    correct_mediums = error_df[(error_df['True_Target'] == 2) & (error_df['Pred_Target'] == 2)].copy()

    print(f" Total True Mediums predicted as Low: {len(medium_as_low)}")

    if len(medium_as_low) > 0:
        csv_name = f"error_analysis_medium_predicted_as_low.csv"
        medium_as_low.to_csv(submission_path + csv_name, index=False)
        print(f"Saved {len(medium_as_low)} error rows to {csv_name} ")

    # (Optional) Save ALL errors just in case you want to look at everything
    all_errors = error_df[error_df['True_Target'] != error_df['Pred_Target']]
    all_errors.to_csv(submission_path + f"error_analysis_ALL_mistakes.csv", index=False)

    # The Country Trap Check
    country_cols = [c for c in error_df.columns if 'country' in c.lower()]
    if country_cols:
        print("FAILED MEDIUMS BY COUNTRY")
        # Handles both one-hot encoded and raw string formats
        if error_df[country_cols[0]].dtype == 'object':
            print(medium_as_low[country_cols[0]].value_counts())
        else:
            print(medium_as_low[country_cols].sum())

    print("="*50 + "\n")

## 3.2. MODEL PIPELINE SETUP

In [ ]:
# Setup Data
le = LabelEncoder()

#ecode the targets
y_enc = le.fit_transform(train_data['Target']) # The Numbers [0, 1, 2] #{High:0, Low:1, Medium:2}
print(np.unique(y_enc, return_counts=True))

#drop the target and ID identifier
X_train = train_data.drop(columns=['Target', 'ID'])

#stratify column by target and country during split
stratify_col = train_data['country'].astype(str) + '_' + train_data['Target'].astype(str)

In [ ]:
def make_ohe_encoder():
    """
    create the one hot encoding function
    """

    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

In [ ]:
def custom_smote_ratios(y):
    """
    Dynamically maps SMOTE ratios based on class frequencies.
    """
    target_stats = Counter(y)

    # Sort the classes by how many rows they have (Highest count to lowest count)
    sorted_classes = sorted(target_stats.items(), key=lambda item: item[1], reverse=True)
    #returns [(np.int64(1), 6280 (low)), (np.int64(2), 2868(medium)), (np.int64(0), 470(high))]

    # Extract the dynamic labels based on their rank
    low_label = sorted_classes[0][0]     # The most frequent class
    medium_label = sorted_classes[1][0]  # The second most frequent class
    high_label = sorted_classes[2][0]    # The least frequent class

    majority_count = sorted_classes[0][1]

    # Apply our 100% / 60% / 25% rule using the dynamically found labels
    return {
        low_label: target_stats[low_label],           # Keep original
        medium_label: int(majority_count * 0.60),     # Pad to 60% of Majority
        high_label: int(majority_count * 0.25)        # Pad to 25% of Majority
    }

In [ ]:
def make_full_pipeline(model, drop_cols=None, use_patterns=True, use_selection=False, use_smote=True):
    """
    Using the custom transformers to create the pipeline for preprocess, feature engineering, correlation checks
    """
    # 1. Define OHE Block (Must be Dense for Pandas output)
    ohe_block = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), selector(dtype_include=np.number)),
            ("cat", Pipeline(steps=[
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", make_ohe_encoder())
            ]), selector(dtype_include=["object", "category", "string"])) # Added string type safety
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )
    ohe_block.set_output(transform="pandas")

    # 2. Build Steps
    steps = [
        ("fhicleaner", FHIDataCleaner()),         #Step 1: Fix typos & Cap Outliers
        ('base_signals', FHIBaseSignals()),     # Step 2: Ratios, text logic, logs
        ('adv_signals', FHIAdvancedSignals()),  # Step 3: Z-scores, Imputation, Clusters

        # Convert all remaining strings to numbers before doing advanced math
        ("ohe", ohe_block),

        # Sanitize feature names  after OHE
        ("sanitizer", FeatureNameSanitizer()),

        # columns correlation checks
        ("cleaner", DataCleaner(correlation_threshold=0.90, variance_threshold=1e-4)),
    ]

    #  clustering  applied to numeric data
    if use_patterns:
        steps.append(('patterns', UnsupervisedPatternExtractor(n_clusters=2, use_gmm=True)))

    #optional specific column drops
    if drop_cols:
        steps.append(("drop", DropColumns(drop_cols)))

    #resampling
    if use_smote:
        steps.append(('smote', SMOTE(sampling_strategy=custom_smote_ratios,
                                     random_state=SEED,k_neighbors=5)))

    # feature selection
    if use_selection:
        scout = ExtraTreesClassifier(criterion='entropy',n_estimators=912,max_depth=25,min_samples_split=10,
            min_samples_leaf = 1,random_state=SEED, n_jobs=-1)
        feat_selector = SelectFromModel(estimator=scout, threshold="median")
        feat_selector.set_output(transform="pandas")
        steps.append(('select', feat_selector))

    # Final Model
    steps.append(("model", model))

    return Pipeline(steps)

## 3.3. EARLY STOPPING SETUP FOR TREE MODELS

In [ ]:
def fit_pipeline_with_early_stopping(pipe, X_tr, y_tr, X_va, y_va, model_name):
    """
    Manually unrolls the pipeline to apply SMOTE to training data only,
    and cleanly transforms validation data before applying early stopping.
    Safely bypasses early stopping for models that don't support it (KNN, RF).
    """
    X_tr_prep, y_tr_prep = X_tr, y_tr
    X_va_prep = X_va

    # Unroll all steps EXCEPT the final classifier
    for name, step in pipe.steps[:-1]:
        if hasattr(step, "fit_resample"):
            X_tr_prep, y_tr_prep = step.fit_resample(X_tr_prep, y_tr_prep)
        else:
            X_tr_prep = step.fit_transform(X_tr_prep, y_tr_prep)

        if hasattr(step, "transform"):
            X_va_prep = step.transform(X_va_prep)

    classifier = pipe.steps[-1][1]
    fit_kwargs = {}

    # Setup Early Stopping only for supported Boosting models
    if "LGBM" in model_name:
        from lightgbm import early_stopping, log_evaluation
        fit_kwargs['eval_set'] = [(X_va_prep, y_va)]
        fit_kwargs['callbacks'] = [early_stopping(50, verbose=False), log_evaluation(0)]
    elif "XGBoost" in model_name:
        classifier.set_params(early_stopping_rounds=50)
        fit_kwargs['eval_set'] = [(X_va_prep, y_va)]
        fit_kwargs['verbose'] = False
    elif "CatBoost" in model_name:
        fit_kwargs['eval_set'] = [(X_va_prep, y_va)]
        fit_kwargs['early_stopping_rounds'] = 50
        fit_kwargs['verbose'] = False

    classifier.fit(X_tr_prep, y_tr_prep, **fit_kwargs)
    pipe.steps[-1] = (pipe.steps[-1][0], classifier)

    return pipe

## 3.4. HYPERPARAMETER TUNING SETUP (OPTUNA)

In [ ]:
def tune_models_with_optuna_fast(expanded_candidates, X, y, stratify_col, n_trials=20, random_state=SEED):
    trained_models = {}
    rows = []

    # 1. Create the single validation split
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=0.2, stratify=stratify_col, random_state=random_state
    )

    for name, (pipe, param_grid) in expanded_candidates.items():
        print(f" Optuna Tuning for {name} ({n_trials} trials)...")

        def objective(trial):
            trial_params = {}
            for param_name, param_values in param_grid.items():
                if isinstance(param_values, list):
                    trial_params[param_name] = trial.suggest_categorical(param_name, param_values)
                elif hasattr(param_values, 'dist') and param_values.dist.name == 'randint':
                    if len(param_values.args) >= 2:
                        low = param_values.args[0]
                        high = param_values.args[1] - 1
                    else:
                        low = param_values.kwds['low']
                        high = param_values.kwds['high'] - 1
                    trial_params[param_name] = trial.suggest_int(param_name, low, high)
                elif hasattr(param_values, 'dist') and param_values.dist.name == 'uniform':
                    if len(param_values.args) >= 2:
                        loc = param_values.args[0]
                        scale = param_values.args[1]
                    else:
                        loc = param_values.kwds['loc']
                        scale = param_values.kwds['scale']
                    trial_params[param_name] = trial.suggest_float(param_name, loc, loc + scale)
                else:
                    trial_params[param_name] = trial.suggest_categorical(param_name, list(param_values))

            # 2. Clone and Apply params
            trial_pipe = clone(pipe)
            trial_pipe.set_params(**trial_params)

            # 3. Fit with EARLY STOPPING
            try:
                trial_pipe = fit_pipeline_with_early_stopping(trial_pipe, X_tr, y_tr, X_va, y_va, model_name=name)
                preds = trial_pipe.predict(X_va)
                score = f1_score(y_va, preds, average='weighted')

                # EXTRACT AND SAVE THE BEST ITERATION TO OPTUNA 🚀
                classifier = trial_pipe.steps[-1][1]
                best_iter = None
                if hasattr(classifier, 'best_iteration_'):   # LGBM
                    best_iter = classifier.best_iteration_
                elif hasattr(classifier, 'best_iteration'):  # XGBoost
                    best_iter = classifier.best_iteration
                elif hasattr(classifier, 'tree_count_'):     # CatBoost
                    best_iter = classifier.tree_count_

                if best_iter is not None:
                    trial.set_user_attr('best_iteration', best_iter)

            except Exception as e:
                print(f"Trial failed: {e}")
                score = 0

            del trial_pipe
            gc.collect()
            return score

        # 4. Run the Study
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", study_name=name)
        study.optimize(objective, n_trials=n_trials)

        best_score = study.best_value
        best_params = study.best_params
        print(f" Best {name} F1: {best_score:.4f}")
        print(f" Best params for : {study.best_params}")

        # 5. retrain best model on full data
        print(f" Retraining best {name} on full 100% data...")
        best_pipe = clone(pipe)
        best_pipe.set_params(**best_params)

        # apply early stopping cap on full refit
        best_iter = study.best_trial.user_attrs.get('best_iteration', None)
        if best_iter is not None:
            # We add 10% more trees because we are training on 20% more data
            optimal_trees = int(best_iter * 1.1)
            classifier_step = best_pipe.steps[-1][1]

            if "CatBoost" in name:
                classifier_step.set_params(iterations=optimal_trees)
            else:
                classifier_step.set_params(n_estimators=optimal_trees)

            print(f"Locked optimal tree count at: {optimal_trees}")

        best_pipe.fit(X, y)

        trained_models[name] = best_pipe
        rows.append({
            "Model": name,
            "BestF1_CV": best_score,
            "BestParams": str(best_params)
        })

    del X_tr, X_va, y_tr, y_va
    gc.collect()

    models_df = pd.DataFrame(rows).sort_values("BestF1_CV", ascending=False).reset_index(drop=True)
    return models_df, trained_models

## 3.5. MODEL ENSEMBLE WEIGHTS

In [ ]:
def find_ensemble_weights(val_probs_dict, y_va_true, n_trials=200):
    """
    val_probs_dict: A dictionary of probabilities predicted on the VALIDATION set.
                    e.g., {'LGBM': lgbm_val_probs, 'CatBoost': cat_val_probs}
    y_va_true: The true numeric target labels for the validation set.
    """
    print("\n🔬 Initiating Optuna Search for best ensemble Weights...")
    model_names = list(val_probs_dict.keys())
    prob_arrays = list(val_probs_dict.values())

    def objective(trial):
        # Optuna suggests a random weight between 0.0 and 1.0 for each model
        weights = [trial.suggest_float(f'w_{name}', 0.0, 1.0) for name in model_names]

        # Normalize the weights so they perfectly add up to 1.0
        weights = np.array(weights)
        if np.sum(weights) == 0:
            return 0.0
        weights /= np.sum(weights)

        # Blend the probabilities using the new weights
        blended_probs = np.zeros_like(prob_arrays[0])
        for w, p in zip(weights, prob_arrays):
            blended_probs += w * p

        # Argmax to get the final predictions
        val_preds = np.argmax(blended_probs, axis=1)

        # Return the Weighted F1 Score (Optuna will try to maximize this)
        return f1_score(y_va_true, val_preds, average='weighted')

    # Suppress Optuna's text warnings
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # Run the study
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    # Print the results
    print(f"Best Validation F1 Reached: {study.best_value:.5f}")

    best_weights = study.best_params
    total = sum(best_weights.values())

    final_weights = {}
    for name in model_names:
        normalized_w = best_weights[f'w_{name}'] / total
        final_weights[name] = normalized_w
        print(f"   > {name}: {normalized_w:.4f}")

    return final_weights

In [ ]:
def save_ensemble_weights(weights_dict, filename):
    """Saves the dictionary of Optuna weights to a human-readable JSON file."""
    # Ensure the directory exists if you are saving to a specific folder
    os.makedirs(os.path.dirname(filename) if os.path.dirname(filename) else '.', exist_ok=True)

    with open(filename, 'w') as f:
        json.dump(weights_dict, f, indent=4)

In [ ]:
def load_ensemble_weights(filename):
    """Loads the dictionary of Optuna weights from a JSON file."""
    if not os.path.exists(filename):
        print(f"{filename} not found. Returning None.")
        return None

    with open(filename, 'r') as f:
        weights_dict = json.load(f)
    print(f"Loaded weights from {filename}:\n{json.dumps(weights_dict, indent=2)}")
    return weights_dict

## 3.6. SAVING THE FEATURE LIST

In [ ]:
def save_pipeline_features(fitted_pipeline, features_path, model_name="model"):
    """
    Extracts and saves the exact feature names seen by the final classifier in a Scikit-Learn Pipeline.
    """
    # Grab the final step of the pipeline (the actual model: LightGBM, XGBoost, etc.)
    classifier = fitted_pipeline.steps[-1][1]

    feature_names = None

    # Check common attribute names for different libraries
    if hasattr(classifier, 'feature_names_in_'):       # Scikit-Learn / XGBoost
        feature_names = classifier.feature_names_in_.tolist()
    elif hasattr(classifier, 'feature_name_'):         # LightGBM
        feature_names = classifier.feature_name_
    elif hasattr(classifier, 'feature_names_'):        # CatBoost
        feature_names = classifier.feature_names_

    if feature_names is not None:
        filename = features_path + f"features_used_{model_name}.json"
        with open(filename, 'w') as f:
            json.dump(feature_names, f, indent=4)
        print(f" Saved {len(feature_names)} feature names to {filename}")
        return feature_names
    else:
        print(f" Could not automatically extract feature names for {model_name}.")
        return None

## 3.7. MODELS DICTIONARY

In [ ]:
candidates = {
    "CatBoost": (
        #base model
        CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric='TotalF1:average=Macro',
            iterations=884,
            learning_rate=0.03,
            l2_leaf_reg=3,
            depth=None,
            verbose=0,
            random_state=SEED,
        ),
        #expanded param grid for tuning
        {
            "model__depth": [4, 6, 8, 10],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
            "model__l2_leaf_reg": [1, 3, 5, 10],
            "model__iterations": randint(200, 1500)
        }
    ),
    "XGBoost": (
        #base model
        XGBClassifier(
            objective='multi:softprob',
            eval_metric='mlogloss',
            n_estimators=400,
            learning_rate=0.03,
            max_depth=8,
            random_state=SEED,
            subsample=0.8,
            n_jobs=-1
        ),
        #expanded param grid for tuning
        {
            "model__max_depth": [3, 4, 6, 8, 10],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
            "model__reg_lambda": [0.1, 1, 5, 10, 20],
            "model__n_estimators": randint(200, 1500),
            "model__subsample": [0.6, 0.8, 1.0],
            "model__colsample_bytree": [0.6, 0.8, 1.0]
        }
    ),

    "RandomForest": (
        #base model
        RandomForestClassifier(
            criterion='entropy',
            n_estimators=500,
            max_depth=None,
            n_jobs=-1,
            random_state=SEED,
            warm_start=False
        ),
        #expanded param grid for tuning
        {
            "model__n_estimators": randint(200, 1500),
            "model__max_depth": [None, 2,4,6,8,10, 15, 20, 25],
            "model__min_samples_split": [2, 5, 10, 15],
            "model__min_samples_leaf": [1, 2, 4, 8]
        }
    ),
    "ExtraTrees": (
        #base model
        ExtraTreesClassifier(
            criterion='entropy',
            n_estimators=912,
            max_depth=25,
            min_samples_split=10,
            min_samples_leaf = 1,
            random_state=SEED,
            n_jobs=-1
        ),
        #expanded param grid for tuning
        {
            "model__n_estimators": randint(200, 1500),
            "model__max_depth": [None, 2,4,6,8,10, 15, 20, 25],
            "model__min_samples_split": [2, 5, 10, 15],
            "model__min_samples_leaf": [1, 2, 4, 8]
        }
    ),
}

## 3.8. RUN THE PIPELINE

In [ ]:
def run_fhi_pipeline(candidates, X, y, stratify_col, test_df, drop_cols, le_target,hardcoded_weights,model_path,ensemble_weights_path,features_path, do_tuning=False, n_iter=5):
    """
    Executes an end-to-end ML pipeline for the FHI dataset, handling
    training, optional hyperparameter tuning, validation, and final ensemble inference.

    Args:
        candidates (dict): A dictionary mapping model names to a tuple of (base_model, param_distributions).
        X (pd.DataFrame): The raw training feature dataset.
        y (np.array or pd.Series): The encoded target labels for the training set.
        stratify_col (np.array or pd.Series): The column to use for stratifying the train/val split (usually the target).
        test_df (pd.DataFrame): The raw test dataset to generate final predictions for.
        drop_cols (list): List of column names to safely drop from the data before training.
        le_target (LabelEncoder): The fitted label encoder used to translate numeric predictions back to strings.
        do_tuning (bool): If True, triggers Optuna hyperparameter tuning. If False, runs a fast 80/20 validation split.
        n_iter (int): Number of Optuna trials to run if do_tuning is True. Defaults to 5.

    Returns:
        tuple: (submission_df, trained_models_dict, leaderboard_df)
    """

    # ==========================================
    # 1. MEMORY OPTIMIZATION & PREPARATION
    # ==========================================
    if drop_cols is None:
        drop_cols = []

    # Safely drop irrelevant columns (like ID) to prevent data leakage or ML crashes
    X_clean = X.drop(columns=drop_cols, errors='ignore').copy()
    X_test_clean = test_df.drop(columns=['ID'] + drop_cols, errors='ignore').copy()

    # Downcasting numerical data to float32  to reduce RAM usage,
    for col in X_clean.select_dtypes(include=['float64']).columns:
        X_clean[col] = X_clean[col].astype('float32')
        X_test_clean[col] = X_test_clean[col].astype('float32')

    trained_models = {}
    models_df = pd.DataFrame()

    # ==========================================
    # 2. EXPAND CANDIDATES (PIPELINE GENERATION)
    # ==========================================
    # For every base model provided, we generate two versions:
    # 1. Standard: Uses basic features (use_patterns=False).
    # 2. Patterns: Uses advanced unsupervised pattern extraction (use_patterns=True).
    expanded_candidates = {}
    for name, (base_model, param_dist) in candidates.items():
        # Force multi-threading to speed up tree construction
        if hasattr(base_model, "set_params"):
            try:
                base_model.set_params(thread_count=4)
            except:
                pass

        expanded_candidates[f"{name}_Standard"] = (
            make_full_pipeline(base_model, use_patterns=False, use_smote=True),
            param_dist
        )
        expanded_candidates[f"{name}_Patterns"] = (
            make_full_pipeline(base_model, use_patterns=True, use_smote=True),
            param_dist
        )

    # ==========================================
    # PATH A: TUNING MODE
    # ==========================================
    if do_tuning:
        print(f"Tuning Enabled (n_iter={n_iter})...")
        # Hands execution over to Optuna to find the mathematically optimal hyperparameters
        leaderboard, tuned_models = tune_models_with_optuna_fast(
            expanded_candidates, X_clean, y, stratify_col, n_trials=n_iter
        )
        leaderboard_df = leaderboard
        trained_models = tuned_models

    # ==========================================
    # PATH B: FAST EVALUATION & ENSEMBLING WITH THE DEFAULT PASSED PARAMETERS
    # ==========================================
    else:
        print("Tuning Disabled")
        rows = []
        val_probs_dict = {}  # Stores out-of-fold probabilities to train the final ensemble

        # Create a strict 80/20 split, stratifying by target to ensure class balance is maintained
        train_idx, val_idx = train_test_split(
            np.arange(len(X_clean)),
            test_size=0.2,
            stratify=stratify_col,
            random_state=SEED
        )

        for name, (pipe, _) in expanded_candidates.items():
            print(f"Evaluating {name}...")
            gc.collect() # Manually clear RAM before training a new model

            X_tr, X_va = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
            y_tr, y_va = y[train_idx], y[val_idx]

            try:
                fold_pipe = clone(pipe)

                # Fit the model using Early Stopping to automatically halt training when validation scores stop improving
                fold_pipe = fit_pipeline_with_early_stopping(fold_pipe, X_tr, y_tr, X_va, y_va, model_name=name)

                # Extract the probability distributions (confidence levels) rather than hard class predictions
                val_probs = fold_pipe.predict_proba(X_va)
                val_probs_dict[name] = val_probs
                preds = np.argmax(val_probs, axis=1)

                score = f1_score(y_va, preds, average='weighted')
                print(f"{name} Base Weighted F1: {score:.4f}")

                # Extract the exact number of trees built before Early Stopping triggered
                classifier = fold_pipe.steps[-1][1]
                best_iter = getattr(classifier, 'best_iteration_',
                            getattr(classifier, 'best_iteration',
                            getattr(classifier, 'tree_count_', None)))

                del fold_pipe, X_tr, preds

            except Exception as e:
                print(f"Error in {name}: {e}")
                score = 0
                best_iter = None

            # Log model performance
            rows.append({"Model": name, "BestF1_CV": score})

            # --- ANTI-OVERFIT REFITTING ---
            # and retrain the model on 100% of the data using the optimal number of trees (best_iter),
            print(f"Refitting {name} on full data...", end=" ")
            gc.collect()

            full_pipe = clone(pipe)
            if best_iter is not None:
                # Add a 10% buffer to the tree count since we are using 20% more training data
                optimal_trees = int(best_iter * 1.1)

                if optimal_trees <= 0:
                    optimal_trees = 100 # Fallback to a safe default
                    print(f"[Warning: best_iter was 0. Using fallback {optimal_trees}]...", end=" ")
                else:
                    print(f"[Locked at {optimal_trees} trees]...", end=" ")

                full_classifier = full_pipe.steps[-1][1]
                if "CatBoost" in name: full_classifier.set_params(iterations=optimal_trees)
                else: full_classifier.set_params(n_estimators=optimal_trees)
                print(f"[Locked at {optimal_trees} trees]...", end=" ")

            full_pipe.fit(X_clean, y)

            save_pipeline_features(full_pipe, features_path, model_name=name)
            trained_models[name] = full_pipe

        # Rank all models from best to worst based on Validation F1
        models_df = pd.DataFrame(rows).sort_values("BestF1_CV", ascending=False).reset_index(drop=True)
        #save the summary dataframe

        # ==========================================
        # THE ENSEMBLE ENGINE & ERROR ANALYSIS
        # ==========================================
        print("\n" + "="*40)

        # 1. ENSEMBLE DILUTION PREVENTION
        # We only blend the top K models.
        K = 3

        # Get the names of the top K models from the sorted models dataframe
        top_k_names = models_df.head(K)['Model'].tolist()
        print(f"ensembling Top {K} Models: {top_k_names}")

        # Isolate the validation probabilities of only the best models
        top_k_val_probs_dict = {name: val_probs_dict[name] for name in top_k_names}

        if hardcoded_weights is not None:
            print("Using hardcoded ensemble weights...")
            ensemble_weights = hardcoded_weights
            save_ensemble_weights(ensemble_weights, filename=ensemble_weights_path)
        else:
            # Use Optuna to find the mathematically perfect ratio/weight for blending these specific models
            ensemble_weights = find_ensemble_weights(top_k_val_probs_dict, y_va, n_trials=250)
            save_ensemble_weights(ensemble_weights, filename=ensemble_weights_path)
            print(f"Using weights: {ensemble_weights}")

        # 2. BLEND VALIDATION PROBABILITIES
        # Apply the Optuna-discovered weights to the validation probabilities to see our final ensemble score
        blended_val_probs = np.zeros_like(list(top_k_val_probs_dict.values())[0])
        for name, p in top_k_val_probs_dict.items():
            blended_val_probs += ensemble_weights[name] * p

        val_preds_ensemble = np.argmax(blended_val_probs, axis=1)
        ensemble_score = f1_score(y_va, val_preds_ensemble, average='weighted')

        print(f"ENSEMBLE VALIDATION SCORE: {ensemble_score:.4f}")
        print(classification_report(y_va, val_preds_ensemble))
        print(confusion_matrix(y_va, val_preds_ensemble))

        print("="*40 + "\n")

        print("\n--- F1-Score Breakdown by Country ---")

        # Check if 'country' exists as a single column (Label Encoded or Raw)
        #print(X_va.columns)
        if 'country' in X_va.columns:
            for c in sorted(X_va['country'].unique()):
                mask = X_va['country'] == c
                if mask.sum() > 0:
                    c_f1 = f1_score(y_va[mask], val_preds_ensemble[mask], average='weighted')
                    print(f" > Country {c} (Support: {mask.sum():>4}): {c_f1:.4f}")
        else:
            print(" [!] 'country' column not found in X_va. (Was it dropped or One-Hot Encoded?)")
        # Pass the ensemble's errors to the diagnostic script for debugging
        perform_error_analysis(X_va, y_va, val_preds_ensemble, blended_val_probs)

    # ==========================================
    # --- FINAL INFERENCE: WEIGHTED SOFT VOTING ---
    # ==========================================
    print("Generating Final Ensemble Submission...")

    # Create a blank canvas to store the final Test Set probabilities
    n_samples = len(X_test_clean)
    n_classes = len(np.unique(y))
    blended_test_probs = np.zeros((n_samples, n_classes))

    # --- WEIGHT HANDLING ---
    # If we bypassed Path B (Ensemble Engine) due to Tuning Mode, we create equal fallback weights
    #the best strategy if tuning is enabled is to get the best params and manually change them in the
    #candidate params and rerun with tuning disabled so as to use top k models only

    if do_tuning:
        print("   Using Equal Weights for Tuned Models (Simple Soft Voting)")
        weight = 1.0 / len(trained_models)
        ensemble_weights = {name: weight for name in trained_models.keys()}
        ensemble_score = models_df['BestF1_CV'].max() if not models_df.empty else 0.0

    # Collect predictions from every fitted model on the test Set
    for name, model in trained_models.items():

        #save the trained models
        filepath = os.path.join(model_path, f"{name}.joblib")
        joblib.dump(model, filepath)
        print(f"Saved: {filepath}")

        # Only use models that Optuna assigned >1% weight to contribute
        if name in ensemble_weights and ensemble_weights[name] > 0.01:
            print(f"   > Extracting {name} (Weight: {ensemble_weights[name]:.3f})")

            # Predict test probabilities and scale them by their assigned weight
            test_probs = model.predict_proba(X_test_clean)
            blended_test_probs += test_probs * ensemble_weights[name]

    # Resolve the final class by picking the highest weighted probability (Argmax)
    final_preds = np.argmax(blended_test_probs, axis=1)

    # Translate integer predictions (0, 1, 2) back into strings ("High", "Low", "Medium")
    final_labels = le_target.inverse_transform(final_preds)

    final_preds_df = pd.DataFrame({
        "ID": test_df["ID"],
        "Target": final_preds
    })

    submission = pd.DataFrame({
        "ID": test_df["ID"],
        "Target": final_labels
    })

    # Save the final file dynamically appended with the local validation score
    sub_name = submission_path + f"submission_Ensemble_Weighted_F1_{ensemble_score:.4f}.csv"
    submission.to_csv(sub_name, index=False)
    print(f"Saved {sub_name}")

    return submission, trained_models, models_df

In [ ]:
# ensemble_weights = {
#     'RandomForest_Patterns': 0.6984,
#     'ExtraTrees_Standard': 0.0005,
#     'ExtraTrees_Patterns': 0.3011
# }
submission_raw, models_raw, models_results_raw =run_fhi_pipeline(
        candidates, X=X_train, y=y_enc, stratify_col=y_enc,
    test_df=test_data, drop_cols=[], le_target=le, hardcoded_weights=None,
        model_path = raw_models_path,ensemble_weights_path= submission_path+ "raw_ensemble_weights.json",
        features_path = raw_models_path,
        do_tuning=False,
        )

In [ ]:
submission_raw['Target'].value_counts()

In [ ]:
display(models_results_raw)
#save the results
models_results_raw.to_csv(submission_path + f"models_results_raw.csv",index=False)

## 3.9.CLEANING NOISY LABELS

In this section, we use the opensource package Cleanlab to clean noisy labels(https://pypi.org/project/cleanlab/)

In [ ]:
def purify_and_blend_predictions(
    X_train,
    y_enc,
    test_data,
    candidates,
    le,
    models_raw,
    models_results_raw,
    make_full_pipeline_fn,
    run_pipeline_fn,
    blend_weight_raw=0.50,
    seed=SEED
):
    """
    Executes a Confident Learning (Label Purification) pipeline using Cleanlab.

    This function trains a conservative 'Judge' model to identify mathematically improbable labels (surveyor typos) via Out-Of-Fold probabilities. It drops the
    disputed rows, retrains the entire ML pipeline on the purified data, and blends the resulting predictions with the original 'Raw' model predictions to create a
    highly robust, hedged final submission.

    Args:
        X_train (pd.DataFrame): The raw training features.
        y_enc (np.array): The encoded training labels.
        test_data (pd.DataFrame): The unseen Kaggle test dataset.
        candidates (dict): Dictionary of ML models and hyperparameter distributions.
        le (LabelEncoder): Fitted label encoder to inverse-transform the final predictions.
        models_raw (dict): The dictionary of trained models from your baseline/raw run.
        models_results_raw (pd.DataFrame): The leaderboard DataFrame from your baseline run.
        make_full_pipeline_fn (function): Your function to build the ML pipeline.
        run_pipeline_fn (function): Your main pipeline runner function (run_fhi_pipeline).
        blend_weight_raw (float): The percentage of trust to give the Raw model (0.0 to 1.0).
        seed (int): Random state for reproducibility.

    Returns:
        pd.DataFrame: The final blended submission ready for Kaggle.
    """
    print(" INITIATING CLEANLAB LABEL PURIFICATION (DROP STRATEGY)")

    # ==========================================
    # STEP 1: Build a conservative main model
    # ==========================================
    # use LightGBM, keeping max_depth shallow prevents the model from memorizing
    # the noise, forcing it to judge based strictly on the strongest macroscopic signals.
    judge_model = LGBMClassifier(
        n_estimators=300,
        max_depth=5,
        min_child_samples=20,
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    )

    # SMOTE must be False to ensure pure  probability generation
    judge_pipeline = make_full_pipeline_fn(
        model=judge_model,
        use_patterns=False,
        use_selection=False,
        use_smote=False
    )

    # data setup
    X_test_clean = test_data.drop(columns=['ID'], errors='ignore').copy()

    for col in X_train.select_dtypes(include=['float64']).columns:
        X_train[col] = X_train[col].astype('float32')
        X_test_clean[col] = X_test_clean[col].astype('float32')

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    # ==========================================
    # STEP 2: Generate Out-Of-Fold (OOF) Probabilities
    # ==========================================
    print("Generating Out-Of-Fold (OOF) Probabilities...")
    oof_probs = cross_val_predict(
        judge_pipeline,
        X_train,
        y_enc,
        cv=skf,
        method='predict_proba'
    )

    # ==========================================
    # STEP 3: Identify Disputed Rows
    # ==========================================
    print(" Searching for mathematical label noise...")

    label_issues_mask = find_label_issues(
        labels=y_enc,
        pred_probs=oof_probs,
        #return_indices_masks=True
    )

    noisy_count = label_issues_mask.sum()
    print(f"Cleanlab identified {noisy_count} highly disputed labels in the dataset!")

    # Display the exact confusion matrix of the disputed rows
    print("Dispute Distribution (Human Label -- Model's Guess):")
    oof_preds = np.argmax(oof_probs, axis=1)
    disputed_df = pd.DataFrame({'Human': y_enc[label_issues_mask], 'Model': oof_preds[label_issues_mask]})
    print(disputed_df.value_counts())

    # ==========================================
    # STEP 4: The "Drop" Strategy
    # ==========================================
    # Instead of relabeling (which can destroy valid edge-cases), we discard the confusing rows.
    clean_mask = ~label_issues_mask

    X_train_purified = X_train[clean_mask].copy()
    y_enc_purified = y_enc[clean_mask].copy()

    print(f"Dropped {noisy_count} confusing rows. Training size: {len(X_train)} , {len(X_train_purified)}")

    # ==========================================
    # STEP 5: Run Pipeline on Purified Data
    # ==========================================
    sub_clean, models_clean, model_results_clean = run_pipeline_fn(
        candidates,
        X=X_train_purified,
        y=y_enc_purified,
        stratify_col=y_enc_purified,
        test_df=test_data,
        drop_cols=[],
        le_target=le,
        hardcoded_weights=None,
        model_path = clean_models_path,
        ensemble_weights_path=submission_path + f'cleaned_ensemble_weights.json',
        features_path = clean_models_path,
        do_tuning=False
    )

    display(model_results_clean)
    model_results_clean.to_csv(submission_path + f"models_results_clean.csv",index=False)
    # ==========================================
    # STEP 6: Extract Probabilities for Blending
    # ==========================================
    print("Extracting Probabilities for the Hedged Blend...")

    n_samples = len(X_test_clean)
    n_classes = len(np.unique(y_enc))

    raw_probs = np.zeros((n_samples, n_classes))
    clean_probs = np.zeros((n_samples, n_classes))

    raw_optuna_weights = load_ensemble_weights(submission_path + 'raw_ensemble_weights.json')
    clean_optuna_weights = load_ensemble_weights(submission_path + 'cleaned_ensemble_weights.json')

    # 2. Extract Raw Probabilities using Optuna Weights
    print(" Gathering Raw probabilities from the raw models")
    for name, model in models_raw.items():
        if name in raw_optuna_weights and raw_optuna_weights[name] > 0.01:
            weight = raw_optuna_weights[name]
            raw_probs += model.predict_proba(X_test_clean) * weight

    # 3. Extract Clean Probabilities using Optuna Weights
    print(" Gathering Clean probabilities from the cleaned models")
    for name, model in models_clean.items():
        if name in clean_optuna_weights and clean_optuna_weights[name] > 0.01:
            weight = clean_optuna_weights[name]
            clean_probs += model.predict_proba(X_test_clean) * weight

    # ==========================================
    # STEP 7: Execute Probability Fusion
    # ==========================================
    blend_weight_clean = 1.0 - blend_weight_raw
    print(f"Executing the {int(blend_weight_raw*100)}/{int(blend_weight_clean*100)} Probability Fusion...")

    # Mathematical fusion of the probability matrices
    final_blended_probs = (raw_probs * blend_weight_raw) + (clean_probs * blend_weight_clean)

    # Resolve probabilities to discrete class integers, then decode to strings
    final_preds = np.argmax(final_blended_probs, axis=1)
    final_labels = le.inverse_transform(final_preds)

    # Generate final submission
    hedge_submission = pd.DataFrame({
        "ID": test_data["ID"],
        "Target": final_labels
    })

    csv_name = submission_path + f"submission_Pruned_Blend_{int(blend_weight_raw*100)}_{int(blend_weight_clean*100)}.csv"
    hedge_submission.to_csv(csv_name, index=False)
    print(f"Saved hedged submission to {csv_name}")

    return hedge_submission

In [ ]:
#generate blended submission using 50/50 weighting strategy
final_blended_submission = purify_and_blend_predictions(
    X_train=X_train,
    y_enc=y_enc,
    test_data=test_data,
    candidates=candidates,
    le=le,
    models_raw=models_raw,
    models_results_raw=models_results_raw, #baseline results  dataframe
    make_full_pipeline_fn=make_full_pipeline,
    run_pipeline_fn=run_fhi_pipeline,
    blend_weight_raw=0.50, # tweak your blend here
    seed=SEED
)

# 4.OPTIONAL ADD-ON : PARMATER TUNING

**Run this section to tune the models. Once the best parameters are obtained, manually change the default params in the models dictionary and rerun the whole section above this one**

In [ ]:
# tuning only top k models
candidates_tuning = {
    "CatBoost": (
        # base model
        CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric='TotalF1:average=Macro',
            iterations=884,
            learning_rate=0.03,
            l2_leaf_reg=3,
            depth=8,
            verbose=0,
            random_state=SEED,
        ),
        # expanded grid
        {
            "model__depth": [4, 6, 8, 10,13,None],
            "model__learning_rate": np.geomspace(0.01,0.2),
            "model__l2_leaf_reg": [1, 3, 5, 10],
            "model__iterations": randint(200, 2000),
        }
    ),
    # "XGBoost": (
    #     # BASE MODEL
    #     XGBClassifier(
    #         objective='multi:softprob',
    #         eval_metric='mlogloss',
    #         n_estimators=500,
    #         learning_rate=0.05,
    #         max_depth=4,
    #         random_state=SEED,
    #         n_jobs=-1
    #         # Note: XGBoost multiclass doesn't have native auto_class_weights.
    #         # Your pipeline's SMOTE step will handle the imbalance for this!
    #     ),
    #     # EXPANDED PARAM GRID (Fixed parameter names!)
    #     {
    #         "model__max_depth": [3, 4, 6, 8, 10],
    #         "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    #         "model__reg_lambda": [0.1, 1, 5, 10, 20], # This is XGB's L2 Reg
    #         "model__n_estimators": randint(200, 1500),
    #         "model__subsample": [0.6, 0.8, 1.0],
    #         "model__colsample_bytree": [0.6, 0.8, 1.0]
    #     }
    # ),

    # "RandomForest": (
    #     # BASE MODEL
    #     RandomForestClassifier(
    #         criterion='entropy',                # Forces pure minority leaves
    #         #class_weight='balanced_subsample',
    #         n_estimators=300,
    #         max_depth=None,
    #         #class_weight="balanced",
    #         n_jobs=-1,
    #         random_state=SEED,
    #         warm_start=False
    #     ),
    #     # EXPANDED PARAM GRID
    #     {
    #         "model__n_estimators": randint(200, 1500),
    #         "model__max_depth": [None, 2,4,6,8,10, 15, 20, 25],
    #         "model__min_samples_split": [2, 5, 10, 15],
    #         "model__min_samples_leaf": [1, 2, 4, 8]
    #     }
    # ),
    "ExtraTrees": (
        # base model
        ExtraTreesClassifier(
            criterion='entropy',
            n_estimators=912,
            max_depth=25,
            min_samples_split=10,
            min_samples_leaf = 1,
            random_state=SEED,
            n_jobs=-1
        ),
        #expanded
        {
            "model__n_estimators": randint(200, 2000),
            "model__max_depth": [None, 2,4,6,8,10, 15, 20, 25],
            "model__min_samples_split": [2, 5, 10, 15],
            "model__min_samples_leaf": [1, 2, 4, 8]
        }
    ),
}

In [ ]:
#uncomment to run tuning

# _, tuned_trained_models, tuned_results_df = run_fhi_pipeline(
#     candidates_tuning, X=X_train, y=y_enc, stratify_col=y_enc,
#     test_df=test_data, drop_cols=[], le_target=le, do_tuning=True
# )

In [ ]:
end = time.time()
print("Total time Taken",(end-start)/60, 'minutes')